# Multi-Positive InfoNCE Loss — Step-by-Step Walkthrough

This notebook walks through how `build_positive_mask` and `multi_positive_infonce_loss` work
using a small, interpretable dummy batch.

## 0. Setup

In [1]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)
print('torch version:', torch.__version__)

torch version: 2.10.0


## 1. Dummy Batch

We create a batch of **6 samples** with the following (category, country) pairs:

| idx | category  | country |
|-----|-----------|---------|
| 0   | beach     | ID      |  ← same key as idx 1 (both are positives to each other)
| 1   | beach     | ID      |  ← same key as idx 0
| 2   | mountain  | JP      |  ← same key as idx 3
| 3   | mountain  | JP      |  ← same key as idx 2
| 4   | city      | US      |  ← unique → no positives in this batch
| 5   | beach     | US      |  ← unique (beach but different country) → no positives

Positives = samples that share **both** category **and** country.

In [2]:
categories = ['beach',    'beach',    'mountain', 'mountain', 'city',  'beach']
countries  = ['ID',       'ID',       'JP',       'JP',       'US',    'US']

N = len(categories)
D = 8   # small embedding dimension for readability

print(f'Batch size N={N}, embedding dim D={D}')
print()
for i, (cat, cou) in enumerate(zip(categories, countries)):
    print(f'  sample {i}: ({cat}, {cou})')

Batch size N=6, embedding dim D=8

  sample 0: (beach, ID)
  sample 1: (beach, ID)
  sample 2: (mountain, JP)
  sample 3: (mountain, JP)
  sample 4: (city, US)
  sample 5: (beach, US)


## 2. `build_positive_mask` — Step by Step

The mask marks every pair *(i, j)* that shares the same `(category, country)` key,
then removes the diagonal so a sample is never its own positive.

### 2a. Hash keys

Each sample is turned into a single integer by hashing the string `"<category>||<country>"`.

In [3]:
keys = torch.tensor([hash(f"{c}||{r}") for c, r in zip(categories, countries)])

print('keys (one integer per sample):')
for i, (cat, cou, k) in enumerate(zip(categories, countries, keys.tolist())):
    print(f'  sample {i} ({cat}, {cou}): key = {k}')

keys (one integer per sample):
  sample 0 (beach, ID): key = 5477089260805409944
  sample 1 (beach, ID): key = 5477089260805409944
  sample 2 (mountain, JP): key = -1853644112570805529
  sample 3 (mountain, JP): key = -1853644112570805529
  sample 4 (city, US): key = 6764943608093787689
  sample 5 (beach, US): key = 2953919144911052204


### 2b. Pairwise equality matrix

`keys.unsqueeze(0)` is shape `(1, N)` and `keys.unsqueeze(1)` is shape `(N, 1)`.
Broadcasting the `==` operator gives an `(N, N)` boolean matrix where
`mask[i,j] = True` when samples *i* and *j* share the same hash key.

In [4]:
mask_with_diag = (keys.unsqueeze(0) == keys.unsqueeze(1))

print('Pairwise equality matrix (including diagonal):')
print(mask_with_diag.int())  # int for cleaner display

Pairwise equality matrix (including diagonal):
tensor([[1, 1, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0],
        [0, 0, 1, 1, 0, 0],
        [0, 0, 1, 1, 0, 0],
        [0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 1]], dtype=torch.int32)


### 2c. Remove diagonal

A sample is always identical to itself, so we zero out the diagonal to avoid
treating self-similarity as a positive signal.

In [5]:
positive_mask = mask_with_diag.clone()
positive_mask.fill_diagonal_(False)

print('positive_mask (diagonal removed):')
print(positive_mask.int())
print()
print('Number of positives per anchor (row sums):')
print(positive_mask.sum(dim=1).tolist())
print()
print('Samples with at least one positive:', positive_mask.any(dim=1).tolist())

positive_mask (diagonal removed):
tensor([[0, 1, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0]], dtype=torch.int32)

Number of positives per anchor (row sums):
[1, 1, 1, 1, 0, 0]

Samples with at least one positive: [True, True, True, True, False, False]


**Interpretation:**
- Samples 0 & 1 are mutual positives (both `beach/ID`).
- Samples 2 & 3 are mutual positives (both `mountain/JP`).
- Samples 4 & 5 have **no positives** in this batch — they will be excluded from the loss.

## 3. Create Dummy Embeddings

We generate random image and text embeddings and **L2-normalize** them to unit sphere,
matching the expectation in `multi_positive_infonce_loss`.

In [11]:
raw_image  = torch.randn(N, D)
raw_text   = torch.randn(N, D)

image_embeds = F.normalize(raw_image, dim=-1)   # (N, D), unit norm
text_embeds  = F.normalize(raw_text,  dim=-1)   # (N, D), unit norm

print('image_embeds shape:', image_embeds.shape)
print('text_embeds  shape:', text_embeds.shape)
print()
print('Verify unit norms (image):', image_embeds.norm(dim=-1).tolist())
print('Verify unit norms (text): ', text_embeds.norm(dim=-1).tolist())

image_embeds shape: torch.Size([6, 8])
text_embeds  shape: torch.Size([6, 8])

Verify unit norms (image): [1.0, 0.9999999403953552, 1.0, 1.0, 1.0, 1.0]
Verify unit norms (text):  [1.0, 0.9999999403953552, 0.9999999403953552, 1.0, 0.9999999403953552, 1.0]


## 4. `multi_positive_infonce_loss` — Step by Step

We will manually replicate every line of the function so you can inspect
intermediate tensors at each stage.

### 4a. Compute raw logits

Logits are cosine-similarity scores (dot product of unit vectors) scaled by temperature.

- `logits_i2t[i, j]` = how similar **image i** is to **text j**
- `logits_t2i[i, j]` = how similar **text i** is to **image j**

In [12]:
temperature = 0.07

logits_i2t = (image_embeds @ text_embeds.T) / temperature   # (N, N)
logits_t2i = (text_embeds  @ image_embeds.T) / temperature  # (N, N)

print('logits_i2t (image→text cosine sim / temperature):')
print(logits_i2t.round(decimals=2))

logits_i2t (image→text cosine sim / temperature):
tensor([[  6.7800,  -4.9600,  -5.8000,   4.5600,   7.2900,   1.7100],
        [ -5.7300,  -1.0700,   1.3500,  -7.1300,  -4.7200,  -6.8500],
        [  5.1100,  -6.2200,   2.5600,   8.3900,   0.4900,   7.0700],
        [  2.6200,   1.8000,   3.1100,   2.7900, -11.4600,  -2.2000],
        [ -0.7400,   0.4300,   1.4900,  -8.6500,  -3.0300,  -3.3500],
        [  4.9200,  -7.0400,  -1.9600,   8.6500,  -1.5200,  -2.6500]])


### 4b. Log-denominator via logsumexp

`log_denom[i]` = log of the sum of exponentiated logits across **all** j for anchor i.
This is the normalising constant for the softmax distribution.

In [13]:
log_denom_i2t = torch.logsumexp(logits_i2t, dim=1)   # (N,)

print('log_denom (one value per anchor):', log_denom_i2t.tolist())

log_denom (one value per anchor): [7.802494525909424, 1.4344533681869507, 8.65860366821289, 4.0723724365234375, 1.8798574209213257, 8.675700187683105]


### 4c. Log-probabilities

`log_probs[i, j]` = log p(j | anchor i) = logit(i,j) − log_denom(i)

This is equivalent to `F.log_softmax(logits, dim=1)` but written out explicitly.

In [14]:
log_probs_i2t = logits_i2t - log_denom_i2t.unsqueeze(1)   # (N, N)

print('log_probs_i2t:')
print(log_probs_i2t.round(decimals=3))
print()
# Sanity check: exp(log_probs).sum(dim=1) should be ~1 for each row
print('Row-sum of probabilities (should all be ~1.0):')
print(log_probs_i2t.exp().sum(dim=1).tolist())

log_probs_i2t:
tensor([[ -1.0210, -12.7610, -13.5990,  -3.2440,  -0.5140,  -6.0930],
        [ -7.1640,  -2.5020,  -0.0890,  -8.5620,  -6.1500,  -8.2860],
        [ -3.5450, -14.8790,  -6.0990,  -0.2700,  -8.1680,  -1.5840],
        [ -1.4490,  -2.2740,  -0.9600,  -1.2820, -15.5300,  -6.2680],
        [ -2.6230,  -1.4480,  -0.3860, -10.5270,  -4.9140,  -5.2260],
        [ -3.7530, -15.7120, -10.6400,  -0.0240, -10.1960, -11.3270]])

Row-sum of probabilities (should all be ~1.0):
[1.0000001192092896, 1.0, 0.9999996423721313, 0.9999997019767761, 0.9999999403953552, 1.000000238418579]


### 4d. Mask out non-positive log-probs

Multiply `log_probs` by `positive_mask.float()` so only the cells
corresponding to true positives are non-zero.

In [15]:
pos_log_probs_i2t = log_probs_i2t * positive_mask.float()   # (N, N)

print('pos_log_probs_i2t (zeroed out for non-positives):')
print(pos_log_probs_i2t.round(decimals=3))
print()
print('positive_mask for reference:')
print(positive_mask.int())

pos_log_probs_i2t (zeroed out for non-positives):
tensor([[ -0.0000, -12.7610,  -0.0000,  -0.0000,  -0.0000,  -0.0000],
        [ -7.1640,  -0.0000,  -0.0000,  -0.0000,  -0.0000,  -0.0000],
        [ -0.0000,  -0.0000,  -0.0000,  -0.2700,  -0.0000,  -0.0000],
        [ -0.0000,  -0.0000,  -0.9600,  -0.0000,  -0.0000,  -0.0000],
        [ -0.0000,  -0.0000,  -0.0000,  -0.0000,  -0.0000,  -0.0000],
        [ -0.0000,  -0.0000,  -0.0000,  -0.0000,  -0.0000,  -0.0000]])

positive_mask for reference:
tensor([[0, 1, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0]], dtype=torch.int32)


### 4e. Per-anchor loss

For each anchor *i*:
1. Sum the masked log-probs over all *j* → total positive log-probability for anchor *i*.
2. Divide by the number of positives to get a **per-anchor mean**.
3. Negate (we minimise negative log-likelihood).

In [16]:
num_positives  = positive_mask.float().sum(dim=1).clamp(min=1)   # (N,)
per_anchor_loss_i2t = -pos_log_probs_i2t.sum(dim=1) / num_positives  # (N,)

print('num_positives per anchor:', num_positives.tolist())
print()
print('per_anchor_loss_i2t (all anchors):', per_anchor_loss_i2t.tolist())
print()
print('Note: anchors 4 and 5 have 0 positives → their loss value is')
print('computed but will be EXCLUDED from the final mean.')

num_positives per anchor: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

per_anchor_loss_i2t (all anchors): [12.761347770690918, 7.163512706756592, 0.2698240280151367, 0.9600939750671387, -0.0, -0.0]

Note: anchors 4 and 5 have 0 positives → their loss value is
computed but will be EXCLUDED from the final mean.


### 4f. Filter to anchors that have at least one positive and compute mean

In [17]:
has_positive = positive_mask.any(dim=1)   # (N,) bool

print('has_positive:', has_positive.tolist())
print()

loss_i2t = per_anchor_loss_i2t[has_positive].mean()
print(f'loss_i2t (image→text, averaged over anchors with positives): {loss_i2t.item():.6f}')

has_positive: [True, True, True, True, False, False]

loss_i2t (image→text, averaged over anchors with positives): 5.288694


### 4g. Repeat for text→image direction

In [18]:
log_denom_t2i    = torch.logsumexp(logits_t2i, dim=1)
log_probs_t2i    = logits_t2i - log_denom_t2i.unsqueeze(1)
pos_log_probs_t2i = log_probs_t2i * positive_mask.float()
per_anchor_loss_t2i = -pos_log_probs_t2i.sum(dim=1) / num_positives

loss_t2i = per_anchor_loss_t2i[has_positive].mean()
print(f'loss_t2i (text→image, averaged over anchors with positives): {loss_t2i.item():.6f}')

loss_t2i (text→image, averaged over anchors with positives): 5.340356


### 4h. Final symmetric loss

In [19]:
loss = (loss_i2t + loss_t2i) / 2
print(f'Final symmetric multi-positive InfoNCE loss: {loss.item():.6f}')

Final symmetric multi-positive InfoNCE loss: 5.314525


## 5. Cross-Check Against the Real Implementation

In [20]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from model.geotir.model import build_positive_mask, multi_positive_infonce_loss

mask_ref = build_positive_mask(categories, countries)
loss_ref = multi_positive_infonce_loss(image_embeds, text_embeds, mask_ref, temperature)

print('Our manual mask matches:', torch.equal(positive_mask, mask_ref))
print(f'Our manual loss: {loss.item():.6f}')
print(f'Reference loss:  {loss_ref.item():.6f}')
print('Match:', torch.isclose(loss, loss_ref).item())

Our manual mask matches: True
Our manual loss: 5.314525
Reference loss:  5.314525
Match: True


## 6. Intuition Summary

| Concept | What it does |
|---|---|
| **Hash key** | Collapses `(category, country)` into one integer for fast pairwise comparison |
| **positive_mask[i,j]** | `True` when sample *j* is a semantic positive for anchor *i* |
| **logits / temperature** | Low temperature → sharper distribution → harder negatives |
| **log_denom (logsumexp)** | Normalisation constant — denominator of the softmax over ALL *j* |
| **pos_log_probs masked** | Only the probability mass assigned to true positives matters |
| **divide by num_positives** | Prevents anchors with many positives from dominating the gradient |
| **filter has_positive** | Anchors with no in-batch positive are skipped — they carry no signal |
| **symmetric average** | Gradients flow through both the image encoder and the text encoder equally |